### Knowledge Graph builder (deontic, party-centric)

#### 1. Setup — locate repo, load API keys

In [4]:
import sys, os, json
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'server').exists():
    ROOT = ROOT.parent
SERVER = ROOT / 'server'
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

from dotenv import load_dotenv
load_dotenv(SERVER / '.env')

print('ROOT   :', ROOT)
print('OPENAI key set:', bool(os.getenv('OPENAI_API_KEY')))

ROOT   : /home/sante/Documents/FGV/me/SecondPaper
OPENAI key set: True


#### 2. Config — provider, input/output folders

In [5]:
PROVIDER = 'openai'   # only 'openai' (gpt-4.1) is wired; factory is extensible

PARAGRAPHS_DIR = ROOT / 'infra/json/paragraphs'
KG_OUT_DIR     = ROOT / 'infra/json/kg'
KG_OUT_DIR.mkdir(parents=True, exist_ok=True)

available = sorted(p.name for p in PARAGRAPHS_DIR.glob('*.json'))
for i, name in enumerate(available):
    print(i, name)

0 root_BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10_1-Supply_Agreement.json
1 root_BELLICUM_MILTENYI_Supply_Agreement_Summary.json
2 root_EdietsComInc_20001030_10QSB_EX-10_4_2606646_EX-10_4_Co-Branding_Agreement.json
3 root_HealthcentralCom_19991108_S-1A_EX-10_27_6623292_EX-10_27_Co-Branding_Agreement.json
4 root_RitterPharmaceuticalsInc_20200313_S-4A_EX-10_54_12055220_EX-10_54_Development_Agreement.json
5 root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement.json
6 root_TomOnlineInc_20060501_20-F_EX-4_46_749700_EX-4_46_Co-Branding_Agreement.json


#### 3. Load one document's paragraphs

In [6]:
DOC_INDEX = 5   # pick from the list above
DOC_FILE = available[DOC_INDEX]

src = json.load(open(PARAGRAPHS_DIR / DOC_FILE))
paragraphs = src['paragraphs']
doc_id = src['documentId']
print(doc_id)
print(len(paragraphs), 'paragraphs')

root::SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10.16_Affiliate Agreement
52 paragraphs


#### 4. Build the knowledge graph

Chunks the paragraphs, calls the LLM per chunk, and merges parties/clauses
across chunks (entity resolution). One LLM call per chunk — long contracts
take a bit.

In [7]:
from services.graph.knowledge.extraction import build_knowledge_graph
from services.llm.factory import LLMProviderFactory

provider = LLMProviderFactory.create(PROVIDER)
kg = build_knowledge_graph(paragraphs, provider)

NODE_KEYS = ['parties', 'clauses', 'definedTerms',
             'obligations', 'rights', 'prohibitions',
             'conditions', 'references', 'values']
nodes = {key: len(getattr(kg, key)) for key in NODE_KEYS}
edges = Counter(e.type for e in kg.edges)

print(f'{"NODES":<24}{sum(nodes.values()):>6}')
for key, n in nodes.items():
    print(f'  {key:<22}{n:>6}')

print(f'\n{"EDGES":<24}{len(kg.edges):>6}')
for t, n in edges.most_common():
    print(f'  {t:<22}{n:>6}')

# Coverage — a statement with no party never surfaces in any party's view.
statements = [*kg.obligations, *kg.rights, *kg.prohibitions]
no_party = (
    [o for o in kg.obligations if not o.burdenPartyId]
    + [r for r in kg.rights if not r.benefitPartyId]
    + [p for p in kg.prohibitions if not p.burdenPartyId]
)
no_clause = [s for s in statements if not s.clauseId]
total = max(1, len(statements))
semantic = edges['references'] + edges['depends_on'] + edges['supersedes'] + edges['modifies']

print(f'\nstatements with no party : {len(no_party):>4}  ({100 * len(no_party) / total:.0f}%)')
print(f'statements with no clause: {len(no_clause):>4}  ({100 * len(no_clause) / total:.0f}%)')
print(f'clause<->clause edges    : {semantic:>4}  (references + depends_on + supersedes + modifies)')


NODES                      137
  parties                    2
  clauses                   29
  definedTerms              10
  obligations               33
  rights                    15
  prohibitions              13
  conditions                10
  references                12
  values                    13

EDGES                      154
  is_part_of               103
  assigns_obligation_to     26
  grants_right_to           11
  defines                    8
  uses                       4
  modifies                   2

statements with no party :   24  (39%)
statements with no clause:    6  (10%)
clause<->clause edges    :    2  (references + depends_on + supersedes + modifies)


#### 5. Save the KG

In [8]:
out_path = KG_OUT_DIR / DOC_FILE
out_path.write_text(json.dumps(kg.model_dump(), ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', out_path)

saved: /home/sante/Documents/FGV/me/SecondPaper/infra/json/kg/root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement.json


#### 6. Inspect — parties and sample statements

In [9]:
for p in kg.parties:
    print(f'[{p.id}] {p.name!r}  role={p.role!r}  aliases={p.aliases}')
print()
for kind, items in (('obligation', kg.obligations), ('right', kg.rights), ('prohibition', kg.prohibitions)):
    for s in items[:5]:
        print(f'[{s.id}] {kind:11s} burden={s.burdenPartyId} benefit={s.benefitPartyId} clause={s.clauseId}')
        print('    ', s.summary)


[party-1] 'Equidata, Inc.'  role='Service Provider'  aliases=['Company', 'EQUIDATA', 'Equidata']
[party-2] 'National Credit Report.com, LLC'  role='Marketing Affiliate'  aliases=['MARKETING AFFILIATE', 'Marketing Affiliate', 'National Credit Report LLC']

[obligation-1] obligation  burden=party-2 benefit=None clause=clause-1
     Marketing Affiliate must collect all amounts due directly from the Consumer.
[obligation-2] obligation  burden=party-2 benefit=None clause=clause-1
     Marketing Affiliate must bear sole responsibility for non-payment of any fees charged to the Consumer.
[obligation-3] obligation  burden=party-2 benefit=party-1 clause=clause-1
     Marketing Affiliate must pay Equidata compensation as outlined in Exhibit A.
[obligation-4] obligation  burden=party-2 benefit=party-1 clause=clause-1
     Marketing Affiliate must pay billed amounts in full within 30 days from the invoice date.
[obligation-5] obligation  burden=party-2 benefit=party-1 clause=clause-1
     Marketin

#### 7. (Optional) Batch — one KG per contract, for every file

Processes all documents in `infra/json/paragraphs/` at once. Each iteration
builds **one** knowledge graph for a whole contract (the internal chunking is
merged), and writes one KG file per contract to `infra/json/kg/`.

In [10]:
# for f in available:
#     s = json.load(open(PARAGRAPHS_DIR / f))
#     g = build_knowledge_graph(s['paragraphs'], provider)
#     (KG_OUT_DIR / f).write_text(json.dumps(g.model_dump(), ensure_ascii=False, indent=2), encoding='utf-8')
#     n = len(g.obligations) + len(g.rights) + len(g.prohibitions)
#     print(f'{f[:55]:57s} statements={n:4d} parties={len(g.parties)}')
